# Add Dimensions to PHyDAP Runoff Data

## Contents
- [Input Datasets for this Notebook](#input-datasets-for-this-notebook)
- [0. Import required modules](#0-import-required-modules)
- [1. Organize data folders and files](#1-organize-data-folders-and-files)
    - [1.1 Create "Runoff" subfolder](#11-create-runoff-subfolder)
    - [1.2 Create subfolders in "Runoff" subfolder ("Raw", "Interim", "Processed")](#12-create-subfolders-in-runoff-subfolder-raw-interim-processed)
    - [1.3 Move input data files to "Raw" subfolder](#13-move-input-data-files-to-raw-subfolder)
- [2. Declare relevant variables (input/output file paths, field names etc.)](#2-declare-relevant-variables-inputoutput-file-paths-field-names-etc)
- [3. Add dimensions to input PHyDAP file](#3-add-dimensions-to-input-phydap-file)
- [4. Explore output data to ensure everything looks good (optional)](#4-explore-output-data-to-ensure-everything-looks-good-optional)

## Input Datasets for this Notebook

Prior to running this notebook, ```depression_depths_calc.ipynb``` must be ran first. Then, the following file must be downloaded and pasted into the "Data" folder at the root project directory:

- ```PHyDAP_era5_hourly_upland_runoff.nc``` from the [FRDR website](https://www.frdr-dfdr.ca/repo/dataset/7ce4bd7a-4bcc-4f8c-8129-32a691f46c8e)



## 0. Import required modules

In [ ]:
# System
import os
from pathlib import Path
import shutil


# Analysis
from netCDF4 import Dataset
import numpy as np
import pandas as pd


# Visualization
import matplotlib.pyplot as plt

In [ ]:
data_cat = input("Enter the data category: Runoff (r), snowmelt (s), rainfall(p)")

## 1. Organize data folders and files

### 1.1 Create "Runoff" subfolder

In [ ]:
# Set watershed name
WATERSHED_NAME = r"Cypress_New"


# Path for this notebook
NOTEBOOK_PATH = Path.cwd()


# Path for PWA root
PWA_PATH = str(NOTEBOOK_PATH.parent.as_posix())


# Path for base raw data folder
BS_DATA_PATH = PWA_PATH + r"/Data/"


# Specify path for watershed folder
WATERSHED_PATH = BS_DATA_PATH + WATERSHED_NAME


# Path for Runoff folder
if data_cat.lower() == 'r':
    FOLDER_PATH = WATERSHED_PATH + r"/Runoff"
elif data_cat.lower() == 's':
    FOLDER_PATH = WATERSHED_PATH + r"/Snowmelt"
elif data_cat.lower() == 'p':
    FOLDER_PATH = WATERSHED_PATH + r"/Rainfall"
else:
    raise ValueError("Invalid data category. Please enter 'r', 's', or 'p'.")


# Create watershed folder in specified path
os.makedirs(FOLDER_PATH, 
            exist_ok=True) # Do nothing if already exists

### 1.2 Create subfolders in "Runoff" subfolder ("Raw", "Interim", "Processed")

In [ ]:
# Specify paths for depression depths subfolders
NC_RAW_PATH = FOLDER_PATH + r"/Raw/"
NC_INTERIM_PATH = FOLDER_PATH + r"/Interim/"
NC_PROCESSED_PATH = FOLDER_PATH + r"/Processed/"


# Create watershed subfolders in specified paths
SUBFOLDERS_LIST = [NC_RAW_PATH,
                   NC_INTERIM_PATH,
                   NC_PROCESSED_PATH]
for sub in SUBFOLDERS_LIST:
    os.makedirs(sub, exist_ok=True)

### 1.3 Move input data files to "Raw" subfolder

In [ ]:
# Specify source and destination folders before moving files
src = Path(BS_DATA_PATH)
dst = Path(NC_RAW_PATH)


# Move files in base data folder to watershed folder
for file in src.iterdir():
     if file.is_file():
        destination_file = dst / file.name
        if destination_file.exists():
            print(f"Skipped (already exists): {file.name}")
            continue
        shutil.move(str(file), destination_file)

## 2. Declare relevant variables (input/output file paths, field names etc.)

In [ ]:
# Function inputs (eventually will become a function)
path_in = NC_RAW_PATH + input('Enter the input file name: ') # + "/PHyDAP_RDRS21_hourly_runoff.nc"
path_out = NC_PROCESSED_PATH
dim1_name = "time_hrs"
dim2_name = "HYBAS_ID"
var_name = input('Enter the expected variable name: e.g. "cropped_runoff"')
var_units = "mm/h"

## 3. Add dimensions to input PHyDAP file

In [ ]:
# Load nc file
nc_file = Dataset(path_in, "r")

In [ ]:
# Create false latitude dimension
lat = np.array([49.43])
nlat = lat.shape[0]

In [ ]:
## Optional quality checks

# max_val = np.max(nc_file.variables[var_name])

# print(max_val)

# # Examine values and indices before transformation
# max_val_idx = np.where(nc_file.variables[var_name] == max_val)

# print(max_val_idx)

# print(nc_file.variables.keys())

# nc_file.variables['time_hrs'].shape

# nc_file.variables[var_name].shape

In [ ]:
# Get time dimension and store as object
var_data = nc_file.variables[dim1_name][:]
nt = var_data.shape[0]


# Assuming basin ids are not specified

# Get the number of hydrobasins
HYBAS_ID = nc_file.variables[dim2_name][:]
nh = HYBAS_ID.shape[0]


# Convert the netcdf data to a large array
raw_data = nc_file.variables[var_name][:]
flat_data = raw_data.flatten()
reshaped = flat_data.reshape((nt, nlat, nh))

In [ ]:
## Second quality check

# print(np.where(reshaped == max_val))

# reshaped.shape

Visualize timeseries before writing to new file

In [ ]:
import matplotlib.pyplot as plt

# Suppose you want to plot for basin_id = 123456
# basin_id = 123456
# idx = np.where(HYBAS_ID == basin_id)[0][0]  # Find index of basin_id

idx = 0 # testing for first subbasin in dataset

# Plot timeseries for that basin (assuming nlat=1)
plt.figure(figsize=(12, 6))
plt.plot(var_data, reshaped[:, 0, idx])
plt.xlabel("Time (hours)")
plt.ylabel("Runoff (mm/h)")
plt.title(f"Runoff timeseries for HYBAS_ID {basin_id}")
plt.tight_layout()
plt.show()

In [ ]:
# Construct longitude-like index
londim_vals = np.arange(1, nh + 1, dtype=np.float64)


# Define NetCDF output
new_file_name = "rdrs_runoff_wlat.nc"
output_path = path_out + "/" + new_file_name
ncout = Dataset(output_path, "w", format="NETCDF4")


# Define dimensions
ncout.createDimension("lon", nh)
ncout.createDimension("lat", 1)
ncout.createDimension("time", nt)


# Create and assign dimension variables
lon_var = ncout.createVariable("lon", "f8", ("lon",))
lon_var.units = "degrees_east"
lon_var.axis = "X"
lon_var[:] = londim_vals

lat_var = ncout.createVariable("lat", "f8", ("lat",))
lat_var.units = "degrees_north"
lat_var.axis = "Y"
lat_var[:] = lat

time_var = ncout.createVariable("time", "f8", ("time",))
time_var.units = "hours since 1900-01-01 00:00:00"
time_var.axis = "T"
time_var[:] = var_data.astype(np.float64)

In [ ]:
# Define runoff variable
fillvalue = 1e32
runoff_var = ncout.createVariable("runoff", "f4", ("time", "lat", "lon"), fill_value=fillvalue)
runoff_var.long_name = "runoff"
runoff_var.units = var_units


# Write runoff data, in chunks if large
if reshaped.size < 2**31 - 1:
    runoff_var[:, :, :] = reshaped
else:
    for period in range(nt // 100000 + 1):
        start = period * 100000
        end = min(nt, (period + 1) * 100000)
        runoff_var[:, :, start:end] = reshaped[:, :, start:end]


# Finalize
ncout.sync()
ncout.close()
print(f"New file written to: {output_path}")

## 4. Explore output data to ensure everything looks good (optional)

In [ ]:
# Open the NetCDF file
ds = Dataset(output_path, mode="r")

print(ds)                        # Print summary
print(ds.variables.keys())      # List of variable names
print(ds.dimensions.keys())     # List of dimensions

# Check variable attributes
print(ds.variables['runoff'].long_name)
print(ds.variables['runoff'].units)
print(ds.variables['time'].units)


In [ ]:
runoff = ds.variables['runoff'][:]  # shape: (lon, lat, time)
print("Runoff shape:", runoff.shape)

lon = ds.variables['lon'][:]
lat = ds.variables['lat'][:]
time = ds.variables['time'][:]

print("lon count:", len(lon))
print("lat count:", len(lat))
print("time count:", len(time))

In [ ]:
# Overall stats
print("Runoff (mm/h):")
print("  Mean:", np.mean(runoff))
print("  Min:", np.min(runoff))
print("  Max:", np.max(runoff))

In [ ]:
# Average over time for each basin
mean_per_basin = np.mean(runoff, axis=2)[:, 0]  # shape: (lon,) if lat = 


# Create DataFrame of HYBAS_ID and mean runoff
df = pd.DataFrame({
    "HYBAS_ID": lon.astype(int),
    "mean_runoff_mm_per_hr": mean_per_basin
})

df.to_csv(path_out + "/mean_runoff_per_basin.csv", index=False)
print("Exported basin-level runoff summary.")


In [ ]:
nc_file.close()

## Testing files

In [ ]:
nc_data = Dataset('../Data/Calibration/cypress/input/rdrs_runoff_wlat.nc', "r")

In [ ]:
nc_data.variables.keys()  # List of variable names

In [ ]:
nc_data.variables['runoff'][316,0,:].max()

In [ ]:
nc_data.close()